In [21]:
from huggingface_hub import hf_hub_download
import importlib.util

# Descarga el loader unificado (está en cualquier repo)
loader_path = hf_hub_download("Reynier/dga-cnn", "dga_loader.py")
spec = importlib.util.spec_from_file_location("dga_loader", loader_path)
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

name_model="logit"
# Carga cualquier modelo
model, m = mod.load_dga_model(name_model)
results = mod.predict_domains(m, model, ["google.com", "xkr3f9mq.ru"])
print(results)

artifacts.joblib:   0%|          | 0.00/8.43M [00:00<?, ?B/s]

model.py: 0.00B [00:00, ?B/s]

  logit ready.
[{'domain': 'google.com', 'label': 'legit', 'score': 0.0318}, {'domain': 'xkr3f9mq.ru', 'label': 'dga', 'score': 0.966}]


In [22]:
# Para un solo dominio, pasamos el dominio en una lista
results = mod.predict_domains(m, model, ["google.com"])

# Como devuelve una lista, accedemos al primer elemento [0]
prediccion = results[0]

print(f"Dominio: {prediccion['domain']}")
print(f"Etiqueta: {prediccion['label']}")
print(f"Puntaje: {prediccion['score']}")

Dominio: google.com
Etiqueta: legit
Puntaje: 0.0318


In [23]:
import pandas as pd
import time

families = ['bazarbackdoor.gz',
            'bazarbackdoor_v2.gz',
            'bazarbackdoor_v3.gz',
            'bigviktor.gz',
            'bumblebee.gz',
            'ccleaner.gz',
            'dmsniff.gz',
            'enviserv.gz',
            'fosniw.gz',
            'goz.gz',
            'gozi_rfc4343.gz',
            'infy.gz',
            'monerodownloader.gz',
            'newgoz.gz',
            'ngioweb.gz',
            'pandabanker.gz',
            'pizd.gz',
            'reconyc.gz',
            'sharkbot.gz',
            'szribi.gz',
            'torpig.gz',
            'tufik.gz',
            'verblecon.gz',
            'wd.gz',
            'xshellghost.gz',
           ]

runs = 30
chunk_size = 50
offset_legit = 1500

if model is None:
    print("❌ No se pudo cargar el modelo FANCI. Deteniendo evaluación.")
else:
    print("🚀 Iniciando evaluación del modelo FANCI...")

    for family in families:
        print(f"📂 Evaluando familia: {family}")

        dga = pd.read_csv(
            f'/content/drive/My Drive/New_Families/{family}',
            chunksize=chunk_size
        )

        legit = pd.read_csv(
            '/content/drive/My Drive/Familias_Test/legit.gz',
            skiprows=range(1, offset_legit + 1),
            chunksize=chunk_size
        )

        for run in range(runs):
            print(f'  🔄 {run+1:02}/{runs}', end='\r')

            try:
                dfw = pd.concat([dga.get_chunk(), legit.get_chunk()], ignore_index=True)
            except StopIteration:
                print(f"\n⚠️ No hay suficientes datos para completar {family}")
                break

            pred = []
            prob = []
            query_time = []

            for domain_to_check in dfw.domain.values:
                st = time.time()

                try:
                    results = mod.predict_domains(m, model, [domain_to_check])
                    result = results[0]

                    label_value = 1 if result['label'] == 'dga' else 0
                    probability = result['score']

                    pred.append(label_value)
                    prob.append(probability)

                except Exception:
                    pred.append(0)
                    prob.append(0.5)

                query_time.append(time.time() - st)

            dfw['pred'] = pred
            dfw['prob'] = prob
            dfw['query_time'] = query_time

            dfw.to_csv(
                f'/content/drive/My Drive/results/results_{name_model}_{family}_{run}.csv.gz',
                index=False
            )

    print("\n✅ Evaluación completada!")

🚀 Iniciando evaluación del modelo FANCI...
📂 Evaluando familia: bazarbackdoor.gz
📂 Evaluando familia: bazarbackdoor_v2.gz
📂 Evaluando familia: bazarbackdoor_v3.gz
📂 Evaluando familia: bigviktor.gz
📂 Evaluando familia: bumblebee.gz
📂 Evaluando familia: ccleaner.gz
📂 Evaluando familia: dmsniff.gz
📂 Evaluando familia: enviserv.gz
📂 Evaluando familia: fosniw.gz
📂 Evaluando familia: goz.gz
📂 Evaluando familia: gozi_rfc4343.gz
📂 Evaluando familia: infy.gz
📂 Evaluando familia: monerodownloader.gz
📂 Evaluando familia: newgoz.gz
📂 Evaluando familia: ngioweb.gz
📂 Evaluando familia: pandabanker.gz
📂 Evaluando familia: pizd.gz
📂 Evaluando familia: reconyc.gz
📂 Evaluando familia: sharkbot.gz
📂 Evaluando familia: szribi.gz
📂 Evaluando familia: torpig.gz
📂 Evaluando familia: tufik.gz
📂 Evaluando familia: verblecon.gz
📂 Evaluando familia: wd.gz
📂 Evaluando familia: xshellghost.gz
  🔄 30/30
✅ Evaluación completada!


In [24]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Función para calcular FPR y TPR
def fpr_tpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn + 1e-10)
    tpr = tp / (tp + fn + 1e-10)
    return fpr, tpr

families = ['bazarbackdoor.gz', 'bazarbackdoor_v2.gz', 'bazarbackdoor_v3.gz', 'bigviktor.gz', 'bumblebee.gz', 'ccleaner.gz', 'dmsniff.gz', 'enviserv.gz', 'fosniw.gz', 'goz.gz', 'gozi_rfc4343.gz', 'infy.gz', 'monerodownloader.gz', 'newgoz.gz', 'ngioweb.gz', 'pandabanker.gz', 'pizd.gz', 'reconyc.gz', 'sharkbot.gz', 'szribi.gz', 'torpig.gz', 'tufik.gz', 'verblecon.gz', 'wd.gz', 'xshellghost.gz']
runs = 30

# Listas para métricas globales
all_acc, all_acc_std = [], []
all_pre, all_pre_std = [], []
all_rec, all_rec_std = [], []
all_f1, all_f1_std = [], []
all_fpr, all_fpr_std = [], []
all_tpr, all_tpr_std = [], []
all_qt, all_qts = [], []
total_unknowns_global = 0

for family in families:
    acc, pre, rec, f1, fpr, tpr, qt = [], [], [], [], [], [], []
    total_unknowns = 0

    for run in range(runs):
        path = f'/content/drive/My Drive/results/results_{name_model}_{family}_{run}.csv.gz'
        if not os.path.exists(path): continue

        df = pd.read_csv(path)
        y_true = (df['label'] == 'dga').astype(int)
        y_pred = df['pred']

        acc.append(accuracy_score(y_true, y_pred))
        pre.append(precision_score(y_true, y_pred, zero_division=0))
        rec.append(recall_score(y_true, y_pred, zero_division=0))
        f1.append(f1_score(y_true, y_pred, zero_division=0))

        fpr_val, tpr_val = fpr_tpr(y_true, y_pred)
        fpr.append(fpr_val)
        tpr.append(tpr_val)
        if 'query_time' in df.columns: qt.append(df['query_time'].mean())

    if acc:
        print(f'{family.split(".")[0]:15}: '
              f'acc:{np.mean(acc):.2f}±{np.std(acc):.3f} '
              f'f1:{np.mean(f1):.2f}±{np.std(f1):.3f} '
              f'pre:{np.mean(pre):.2f}±{np.std(pre):.3f} '
              f'rec:{np.mean(rec):.2f}±{np.std(rec):.3f} '
              f'FPR:{np.mean(fpr):.2f}±{np.std(fpr):.3f} '
              f'TPR:{np.mean(tpr):.2f}±{np.std(tpr):.3f} '
              f'QT:{np.mean(qt):.5f}±{np.std(qt):.5f}')

        all_acc.append(np.mean(acc))
        all_acc_std.append(np.std(acc))
        all_pre.append(np.mean(pre))
        all_pre_std.append(np.std(pre))
        all_rec.append(np.mean(rec))
        all_rec_std.append(np.std(rec))
        all_f1.append(np.mean(f1))
        all_f1_std.append(np.std(f1))
        all_fpr.append(np.mean(fpr))
        all_fpr_std.append(np.std(fpr))
        all_tpr.append(np.mean(tpr))
        all_tpr_std.append(np.std(tpr))
        all_qt.append(np.mean(qt))
        all_qts.append(np.std(qt))
        total_unknowns_global += total_unknowns

print("\n### ✅ Métricas recolectadas correctamente ###")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
bazarbackdoor  : acc:0.92±0.029 f1:0.92±0.031 pre:0.96±0.035 rec:0.89±0.045 FPR:0.04±0.035 TPR:0.89±0.045 QT:0.00209±0.00035
bazarbackdoor_v2: acc:0.87±0.029 f1:0.86±0.035 pre:0.95±0.039 rec:0.78±0.053 FPR:0.04±0.035 TPR:0.78±0.053 QT:0.00184±0.00017
bazarbackdoor_v3: acc:0.53±0.024 f1:0.18±0.069 pre:0.73±0.185 rec:0.10±0.044 FPR:0.04±0.035 TPR:0.10±0.044 QT:0.00211±0.00032
bigviktor      : acc:0.57±0.028 f1:0.30±0.076 pre:0.83±0.121 rec:0.19±0.056 FPR:0.04±0.035 TPR:0.19±0.056 QT:0.00181±0.00019
bumblebee      : acc:0.93±0.025 f1:0.92±0.026 pre:0.96±0.035 rec:0.90±0.036 FPR:0.04±0.035 TPR:0.90±0.036 QT:0.00225±0.00045
ccleaner       : acc:0.95±0.023 f1:0.95±0.024 pre:0.96±0.033 rec:0.94±0.042 FPR:0.04±0.035 TPR:0.94±0.042 QT:0.00184±0.00018
dmsniff        : acc:0.96±0.040 f1:0.96±0.042 pre:0.96±0.033 rec:0.96±0.068 FPR:0.04±0.035 TPR:0.96±0.068 QT:0.00207±0.

In [25]:
import pandas as pd
import numpy as np

# Crear el DataFrame usando todas las listas de medias y desviaciones
metrics_dict = {
    'Family': [f.split('.')[0] for f in families],
    'Accuracy': all_acc,
    'Acc_std': all_acc_std,
    'Precision': all_pre,
    'Pre_std': all_pre_std,
    'Recall': all_rec,
    'Rec_std': all_rec_std,
    'F1-Score': all_f1,
    'F1_std': all_f1_std,
    'FPR': all_fpr,
    'FPR_std': all_fpr_std,
    'TPR': all_tpr,
    'TPR_std': all_tpr_std,
    'Query_Time_Mean': all_qt,
    'Query_Time_Std': all_qts
}

df_metrics = pd.DataFrame(metrics_dict)

# Calcular fila de promedios globales
global_means = df_metrics.mean(numeric_only=True).to_dict()
global_means['Family'] = 'GLOBAL_MEAN'

df_metrics = pd.concat([df_metrics, pd.DataFrame([global_means])], ignore_index=True)

output_path = f'/content/drive/My Drive/results/metricas_globales_final_{name_model}.csv'
df_metrics.to_csv(output_path, index=False)

print(f"✅ CSV final con todas las desviaciones guardado en: {output_path}")
display(df_metrics)

✅ CSV final con todas las desviaciones guardado en: /content/drive/My Drive/results/metricas_globales_final_logit.csv


,Family,Accuracy,Acc_std,Precision,Pre_std,Recall,Rec_std,F1-Score,F1_std,FPR,FPR_std,TPR,TPR_std,Query_Time_Mean,Query_Time_Std
0,bazarbackdoor,0.922000,0.028682,0.955910,0.035365,0.886000,0.045063,0.918791,0.030621,0.042,0.034775,0.886000,0.045063,0.002092,0.000354
1,bazarbackdoor_v2,0.869667,0.028692,0.950959,0.038901,0.781333,0.053400,0.856309,0.034815,0.042,0.034775,0.781333,0.053400,0.001836,0.000167
2,bazarbackdoor_v3,0.530333,0.023732,0.733678,0.185225,0.102667,0.044342,0.176248,0.069422,0.042,0.034775,0.102667,0.044342,0.002111,0.000320
3,bigviktor,0.572000,0.028213,0.834140,0.121394,0.186000,0.055893,0.299105,0.075678,0.042,0.034775,0.186000,0.055893,0.001813,0.000188
4,bumblebee,0.927333,0.025157,0.956470,0.034957,0.896667,0.036179,0.924955,0.026069,0.042,0.034775,0.896667,0.036179,0.002250,0.000452
5,ccleaner,0.948000,0.023438,0.958626,0.032791,0.938000,0.042379,0.947259,0.024439,0.042,0.034775,0.938000,0.042379,0.001844,0.000181
6,dmsniff,0.959667,0.039706,0.958883,0.033384,0.961333,0.067908,0.958752,0.042323,0.042,0.034775,0.961333,0.067908,0.002072,0.000327
7,enviserv,0.969000,0.018321,0.960029,0.031976,0.980000,0.015492,0.969560,0.017605,0.042,0.034775,0.980000,0.015492,0.001951,0.000313
8,fosniw,0.479000,0.017388,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.042,0.034775,0.000000,0.000000,0.001978,0.000269
9,goz,0.979000,0.017388,0.960744,0.031503,1.000000,0.000000,0.979713,0.016530,0.042,0.034775,1.000000,0.000000,0.002065,0.000340
